# 03 — LoRA supervised fine-tuning

Completion-only SFT teaches Gemma 4 to turn a task-only prompt into the four
scientific work-product sections. The base weights remain frozen and only LoRA
parameters are optimized. Device selection is automatic: **CUDA → MPS → CPU**.

## Configuration

All model, LoRA, training, checkpoint, generation, and Hub settings used below
are explicit variables. Hugging Face uses cached authentication by default.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_from_disk
from IPython.display import JSON, Markdown, display
from trl import SFTConfig, SFTTrainer

from science_course.devices import clear_device_cache, detect_runtime
from science_course.hub import require_hf_namespace
from science_course.modeling import (
    generate_text,
    load_causal_lm,
    load_tokenizer,
    render_prompt,
    render_sft_completion,
    teaching_lora_config,
)
from science_course.versions import require_training_stack

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Runtime and local artifacts
ENABLE_MPS_FALLBACK = True
TOKENIZERS_PARALLELISM = False
if ENABLE_MPS_FALLBACK:
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = str(TOKENIZERS_PARALLELISM).lower()
MODEL_ID = "google/gemma-4-E4B-it"
SFT_DATA = ROOT / "data" / "processed" / "scientific_design_sft"
OUTPUT_DIR = ROOT / "artifacts" / "gemma4-scientific-design-sft"
RESUME_FROM_CHECKPOINT = None

# LoRA
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

# SFT optimization
NUM_TRAIN_EPOCHS = 3
MAX_STEPS = -1
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQUENCE_LENGTH = 1_536
COMPLETION_ONLY_LOSS = True
LOSS_TYPE = "nll"
GRADIENT_CHECKPOINTING = True
GRADIENT_CHECKPOINTING_USE_REENTRANT = False
OPTIMIZER = "adamw_torch"
WARMUP_STEPS = 10
EVAL_STRATEGY = "steps"
EVAL_STEPS = 25
SAVE_STRATEGY = "steps"
SAVE_STEPS = 25
SAVE_TOTAL_LIMIT = None
LOGGING_STEPS = 5
LOGGING_FIRST_STEP = True
REPORT_TO = "none"
RANDOM_SEED = 17
PIN_MEMORY_ON_CUDA_ONLY = True

# Inference demonstration
MAX_NEW_TOKENS = 512
INFERENCE_DO_SAMPLE = False
INFERENCE_TEMPERATURE = 1.0
INFERENCE_TOP_P = 1.0

# Hugging Face publication
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
DATASET_CONFIG_NAME = "scientific_design_sft"
SFT_HF_REPO = "lamm-mit/scientific-sft-grpo-design-sft"
PUSH_TO_HUB = True
HUB_STRATEGY = "all_checkpoints"
HUB_PRIVATE_REPO = False
HUB_ALWAYS_PUSH = True
HF_TOKEN = None
# HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

versions = require_training_stack()
runtime = detect_runtime()
DATALOADER_PIN_MEMORY = (
    runtime.backend == "cuda" if PIN_MEMORY_ON_CUDA_ONLY else True
)
if PUSH_TO_HUB:
    require_hf_namespace(SFT_HF_REPO, token=HF_TOKEN)
display(
    JSON(
        {
            "runtime": runtime.as_dict(),
            "versions": versions,
            "model": MODEL_ID,
            "local_dataset": str(SFT_DATA),
            "hub_dataset": f"{DATASET_HF_REPO}/{DATASET_CONFIG_NAME}",
            "hub_model": SFT_HF_REPO,
        }
    )
)

## 1. Render task-only prompts and structured completions

The tokenizer's chat template formats each prompt. Loss is computed only on the
assistant completion, which contains the four tagged sections.

In [ ]:
if not SFT_DATA.exists():
    raise RuntimeError("Run notebook 01 to create the SFT dataset.")
sft = load_from_disk(SFT_DATA)
if len(sft["train"]) == 0 or len(sft["validation"]) == 0:
    raise RuntimeError("Both SFT train and validation splits must be non-empty.")

tokenizer = load_tokenizer(MODEL_ID, token=HF_TOKEN)

def render_row(row):
    return {
        "prompt": render_prompt(tokenizer, row["prompt"]),
        "completion": render_sft_completion(tokenizer, row["completion"]),
    }

rendered = sft.map(
    render_row,
    remove_columns=sft["train"].column_names,
    desc="Apply the Gemma chat template",
)
display(JSON(rendered["train"][0]))

In [ ]:
lengths = pd.DataFrame(
    {
        split_name: pd.Series(
            [
                len(
                    tokenizer(
                        item["prompt"] + item["completion"]
                    ).input_ids
                )
                for item in split_data
            ]
        )
        for split_name, split_data in rendered.items()
    }
)
lengths.plot.hist(bins=25, alpha=0.65, figsize=(10, 4.5))
plt.axvline(
    MAX_SEQUENCE_LENGTH,
    color="crimson",
    linestyle="--",
    label="training max_length",
)
plt.xlabel("tokens")
plt.title("Structured SFT sequence-length audit")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Attach LoRA and configure the trainer

No CUDA-only quantization dependency is used, so the same path works on CUDA,
Apple MPS, and CPU.

In [ ]:
clear_device_cache(runtime)
model = load_causal_lm(MODEL_ID, runtime, token=HF_TOKEN)
lora = teaching_lora_config(
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
    dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
)

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_length=MAX_SEQUENCE_LENGTH,
    completion_only_loss=COMPLETION_ONLY_LOSS,
    loss_type=LOSS_TYPE,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={
        "use_reentrant": GRADIENT_CHECKPOINTING_USE_REENTRANT
    },
    optim=OPTIMIZER,
    warmup_steps=WARMUP_STEPS,
    eval_strategy=EVAL_STRATEGY,
    eval_steps=EVAL_STEPS,
    save_strategy=SAVE_STRATEGY,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    logging_first_step=LOGGING_FIRST_STEP,
    report_to=REPORT_TO,
    dataloader_pin_memory=DATALOADER_PIN_MEMORY,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=SFT_HF_REPO,
    hub_strategy=HUB_STRATEGY,
    hub_private_repo=HUB_PRIVATE_REPO,
    hub_token=HF_TOKEN,
    hub_always_push=HUB_ALWAYS_PUSH,
    bf16=runtime.trainer_bf16,
    fp16=runtime.trainer_fp16,
    use_cpu=runtime.use_cpu,
    seed=RANDOM_SEED,
)
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=rendered["train"],
    eval_dataset=rendered["validation"],
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()

## 3. Train, evaluate, checkpoint, and publish

Every saved checkpoint and the final adapter are uploaded when publication is
enabled.

In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
if PUSH_TO_HUB:
    hub_result = trainer.push_to_hub(
        commit_message="Complete scientific design SFT training"
    )
    print(f"Published final adapter and all checkpoints: {hub_result}")
display(JSON(train_result.metrics))

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if "loss" in history:
    history.dropna(subset=["loss"]).plot(
        x="step", y="loss", ax=axes[0], color="#315c8c", legend=False
    )
axes[0].set_title("Completion-only SFT loss")
if "eval_loss" in history:
    history.dropna(subset=["eval_loss"]).plot(
        x="step",
        y="eval_loss",
        ax=axes[1],
        color="#d97732",
        legend=False,
    )
axes[1].set_title("Held-out SFT loss")
plt.tight_layout()
plt.show()

## 4. Use the adapter on a new task

Inference needs only a new task. It does not require a paper, reference completion,
hidden rubric, or teacher API call.

In [ ]:
new_task = (
    "Design a self-healing hydrogel for repeated deformation in water. "
    "The material may use reversible physical interactions, but recovery must "
    "not require external heating. Develop several mechanistic strategies, "
    "identify the governing design principles, synthesize the strongest design, "
    "and give a final recommendation."
)
new_messages = [
    {
        "role": "system",
        "content": (
            "Solve the self-contained scientific problem-solving task. Develop "
            "candidate ideas, identify principles, synthesize them, and answer."
        ),
    },
    {
        "role": "user",
        "content": (
            f"SCIENTIFIC PROBLEM-SOLVING TASK\n{new_task}\n\n"
            "Respond with <brainstorm>, <principles>, <synthesis>, and "
            "<answer> in that order, with no text outside the tags."
        ),
    },
]
prompt = render_prompt(tokenizer, new_messages)
response = generate_text(
    trainer.model,
    tokenizer,
    prompt,
    runtime,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=INFERENCE_DO_SAMPLE,
    temperature=INFERENCE_TEMPERATURE,
    top_p=INFERENCE_TOP_P,
)
display(Markdown(f"```text\n{response}\n```"))

## Result

The local adapter is in `artifacts/gemma4-scientific-design-sft/`, and the final
adapter plus every saved checkpoint are published to the configured Hub model
repository. Notebook 04 continues this same adapter with Luna-judged GRPO.